# Testing the hierarchical reserving models on a simulated portfolio (baseline scenario)
Author: Christophe Nozaradan (extension of a previous version by Jens Robben)  
Date: November 2021

## Introduction
This version of this document extends a previous one written by Jens Robben, in which hierarchical GLM and hierarchical GBM are compared in the case of an extreme scenario. In the present version, hierarchical XGB, hierarchical MLP and hierarchical CANN are further considered in the case of a baseline event scenario. 

## Simulating the portfolio
We evaluate the predictive performance of our hierarchical reserving model (hierarchical GLM, hierarchical GBM, hierarchical XGB and hierarchical MLP) on one portfolio simulated along the baseline event scenario. We compare the results with the ones obtained using an aggregate reserving model, namely the classical chain ladder model.

The seed number is:

In [1]:
seed = 1
print(seed)

1



First, we load the required packages.

In [2]:
import pandas as pd
import numpy as np
import hirem as hrm

from importlib import resources as importlib_resources





We simulate one portfolio along the baseline event scenario. The portfolio initially consists of 125 000
 claims, but only the claims that are reported between January 1, 2012 and December 31, 2020 are retained. Claims are tracked over 9 development years since reporting. In addition, we create an extra covariate devYearMonth, representing a simplified version of the interaction effect between dev.year and rep.month.

In [3]:
# The complete portfolio

# simulate from package function
#reserving_data = hrm.simulate_scenario_baseline(seed = seed, n = 125000)

# load from package dataset
my_resources = importlib_resources.files("hirem")
reserving_data = pd.read_csv(my_resources.joinpath("data", "reserving_data.zip"))

reserving_data.head(10)


,Unnamed: 0.1,Unnamed: 0,claim.nr,type,hidden,occ.date,occ.year,occ.year.fact,occ.month,rep.delay,rep.date,rep.year,rep.year.fact,rep.month,settlement.date,settlement.year,dev.year,dev.year.fact,calendar.year,open,settlement,payment,size
0,0,1,1,T2,M,2012-10-13,2012,2012,10,0.219028,2013-01-01,2,2,1,2015-10-24,4,1,1,2,1,0,1,1030.934091
1,1,2,1,T2,M,2012-10-13,2012,2012,10,0.219028,2013-01-01,2,2,1,2015-10-24,4,2,2,3,1,0,1,1353.223321
2,2,3,1,T2,M,2012-10-13,2012,2012,10,0.219028,2013-01-01,2,2,1,2015-10-24,4,3,3,4,1,1,1,517.997112
3,3,4,1,T2,M,2012-10-13,2012,2012,10,0.219028,2013-01-01,2,2,1,2015-10-24,4,4,4,5,0,1,0,0.000000
4,4,5,1,T2,M,2012-10-13,2012,2012,10,0.219028,2013-01-01,2,2,1,2015-10-24,4,5,5,6,0,1,0,0.000000
5,5,6,1,T2,M,2012-10-13,2012,2012,10,0.219028,2013-01-01,2,2,1,2015-10-24,4,6,6,7,0,1,0,0.000000
6,6,7,1,T2,M,2012-10-13,2012,2012,10,0.219028,2013-01-01,2,2,1,2015-10-24,4,7,7,8,0,1,0,0.000000
7,7,8,1,T2,M,2012-10-13,2012,2012,10,0.219028,2013-01-01,2,2,1,2015-10-24,4,8,8,9,0,1,0,0.000000
8,8,9,1,T2,M,2012-10-13,2012,2012,10,0.219028,2013-01-01,2,2,1,2015-10-24,4,9,9,10,0,1,0,0.000000
9,9,10,2,T1,M,2020-09-12,2020,2020,9,0.057495,2020-10-03,9,9,10,2024-10-23,13,1,1,9,1,0,1,315.126043


In [4]:

# Creating the interaction effect
reserving_data['monthDev12'] = reserving_data['rep.month'].astype(str)
reserving_data.loc[reserving_data['dev.year'] > 3, 'monthDev12'] = 'dev.year.over.3'
reserving_data['devYearMonth'] = reserving_data['dev.year'].astype(str) + '-' + reserving_data['monthDev12']

# Observed and prediction data set
observed_data = reserving_data[reserving_data['calendar.year'] <= 9]
prediction_data = reserving_data[reserving_data['calendar.year'] > 9]

## Calibration of the hierarhical reserving models
We first define the weights used in the calibration of the hierarchical reserving model. The purpose of these weights is to replicate the development year distribution from the prediction data set to the observed data set (training data set). Hence, more weight is assigned to claims observations in later development years since reporting. We give a very small weight to the claims observations in the first development year since reporting because the prediction data set does not contain observations for this development year. More details are provided in @hirempaper

In [5]:
# Calculating the weights

# Filter and group the data
reported_claims = observed_data[observed_data['dev.year'] == 1]
reported_claims = reported_claims.groupby('rep.year').size()

# Calculate denominator and numerator
denominator = reported_claims.cumsum().iloc[::-1].iloc[1:]
numerator = reported_claims.iloc[::-1].cumsum().iloc[:-1]

# Calculate weight
weight = np.concatenate(([10**(-6)], numerator.reset_index(drop=True) / denominator.reset_index(drop=True)))

# Create a named series
weight_series = pd.Series(weight, index=[f'dev.year{i+1}' for i in range(9)])
print(weight_series)



dev.year1    0.000001
dev.year2    0.127812
dev.year3    0.290035
dev.year4    0.502695
dev.year5    0.801366
dev.year6    1.257739
dev.year7    2.006846
dev.year8    3.491200
dev.year9    8.073595
dtype: float64



### Hierarchical GLM

The hierarchical GLM consists of the three layer structure: settlement, payment and size. We model the settlement indicator using a binomial regression model with a complementary log link function. The payment indicator will be modeled using a logistic regression model and the payment sizes using a Gamma GLM with log-link function. 

We define the model specifications of each layer. We only train the reserving model on the claim updates that are still open at the beginning of each year and are observed on the evaluation date (`calendar.year <= 9`).


In [6]:
# Model specifications hierarchical GLM
model_glm = hrm.hirem(reserving_data)
model_glm = hrm.split_data(model_glm, observed=lambda x: x[x['calendar.year'] <= 9])  # Example observed data filter)

model_glm = hrm.layer_glm(obj = model_glm, name='settlement', family='binomial(link = cloglog)')
model_glm = hrm.layer_glm(obj = model_glm,name='payment', family='binomial(link = logit)')
model_glm = hrm.layer_glm(obj = model_glm,name='size', family='Gamma(link = log)', filter_func=lambda x: x[x['payment'] == 1])

print(model_glm)

TypeError: list indices must be integers or slices, not str


Next, we perform a covariate selection procedure to select the optimal covariates that are retained in the regression models for each layer. Here, we just list the formulae with the retained covariates, but the details of this covariate selection are provided in @hirempaper.

In [7]:
# Formulae hierarchical GLM layers - model calibration
formula_settle_glm = "settlement ~ type + dev.year.fact"
formula_pay_glm    = "payment ~ settlement + type + devYearMonth"
formula_size_glm   = "size ~ devYearMonth + type + settlement + rep.month"

We fit the hierarchical GLM on the observed portfolio of simulated claims.

In [9]:
# Fitting the hierarchical GLM
model_glm = hrm.fit_hirem(model_glm,
                 weights = weight,
                 weight_var = 'dev.year',
                 balance_var = 'dev.year',
                 formulas = {"settlement": formula_settle_glm, "payment": formula_pay_glm, "size": formula_size_glm},
                 )

print(model_glm)